# 004 — Agentes racionales, entornos y medidas de desempeño

Este notebook reutiliza el mismo núcleo ejecutable que `lab.py`. El objetivo no es
ocultar la implementación, sino separar exploración, ejercicio y solución.

**Evidencia esperada:** resultado JSON, interpretación de una decisión y una
limitación documentada.


## 📖 Resumen de la materia

Un **agente** percibe su entorno con sensores y actúa con actuadores; su **función**
mapea historiales de percepción a acciones. Un **agente racional** elige la acción que
maximiza el **valor esperado de la medida de desempeño** con la información disponible —
racional ≠ omnisciente y racional ≠ perfecto: se juzga la decisión, no el desenlace.

- **PEAS:** especificación previa del entorno de tareas — Performance, Environment,
  Actuators, Sensors. La medida P debe evaluar estados del *entorno*, no del agente
  (medidas hackeables → agentes que hacen trampa literal).
- **Dimensiones del entorno:** observable total/parcial · uni/multiagente ·
  determinista/estocástico · episódico/secuencial · estático/dinámico · discreto/continuo ·
  conocido/desconocido. Determinan qué arquitectura es viable.
- **Taxonomía de agentes:** reflejo simple → reflejo con estado (modelo) → basado en metas
  → basado en utilidad → agente que aprende.

Ejemplo de la teoría: en el mundo de dos casillas con costo de movimiento, un reflejo
simple oscila pagando −1 por viaje; un agente con estado interno espera hasta que la
probabilidad acumulada de suciedad justifique el viaje — la racionalidad depende de P.

In [ ]:
from ai_evolution.labs import run_lab
import json

def show(value):
    print(json.dumps(value, ensure_ascii=False, indent=2))


## Solución de referencia

**Ejercicio 1.** Lo esencial evaluado: (a) P vulnerable = "paquetes movidos por hora"
(incentiva lanzar paquetes); mejor: "paquetes en destino correcto e intactos por hora".
(b) P vulnerable = "ejercicios completados" (incentiva dar ejercicios triviales); mejor:
"progreso medido en evaluaciones independientes". (c) P vulnerable = "tickets cerrados"
(incentiva cerrar sin resolver); mejor: "resolución confirmada por el usuario + tasa de
reapertura".

**Ejercicio 2.** Ajedrez: totalmente observable, multiagente, determinista, secuencial,
estático, discreto, conocido. Póker: parcialmente observable y estocástico (cartas
ocultas). Conducir: parcial, multiagente, estocástico, secuencial, dinámico, continuo,
parcialmente desconocido — el caso más difícil en todas las dimensiones. El laboratorio:
un agente, estocástico controlado por semilla, episódico, estático, discreto, conocido.

**Ejercicio 3.** Con ambas limpias, el alternante paga −1 por viaje sin ganancia extra:
en 5 pasos típicos acumula ~2·5−4 = 6 (limpieza +2/paso menos ~4 movimientos), mientras
el agente con estado hace NoOp y acumula ~10−1. La política óptima depende del costo de
movimiento y de la tasa de re-ensuciamiento — cambiar P cambia al agente óptimo.

**Ejercicio 4.** La política (las reglas de decisión) es la misma; la semilla solo cambia
el entorno muestreado. Un resultado peor con otra semilla NO implica irracionalidad: la
racionalidad se define sobre el valor esperado, no sobre cada desenlace.

In [ ]:
result = run_lab("agent", seed=4)
assert result["kind"] == "agent"
assert result["evidence"]
show(result)


In [ ]:
# Ejercicio 3 — simulación comparada con re-ensuciamiento determinista simplificado
def simula(politica, pasos=5):
    import random
    rng = random.Random(0)
    sucia = {"A": True, "B": True}
    pos, total = "A", 0
    for _ in range(pasos):
        percepcion = (pos, sucia[pos])
        accion = politica(percepcion)
        if accion == "Aspirar":
            sucia[pos] = False
        elif accion in ("Izquierda", "Derecha"):
            pos = "B" if pos == "A" else "A"
            total -= 1
        total += sum(1 for v in sucia.values() if not v)
        for c in sucia:  # re-ensuciamiento p=0.1
            if rng.random() < 0.1:
                sucia[c] = True
    return total

alternante = lambda p: "Aspirar" if p[1] else ("Derecha" if p[0] == "A" else "Izquierda")
visitado = {"A": False, "B": False}
def con_estado(p):
    visitado[p[0]] = True
    if p[1]:
        return "Aspirar"
    if not all(visitado.values()):
        return "Derecha" if p[0] == "A" else "Izquierda"
    return "NoOp"

print("alternante:", simula(alternante), "| con estado:", simula(con_estado))
# El agente con estado evita movimientos sin evidencia de suciedad y rinde más.

## Reflexión

1. En el laboratorio `agent`, identifica qué juega el papel de percepción, acción y medida
   de desempeño en el JSON. ¿La medida evalúa el estado del entorno o el del agente?
   ¿Podría hackearse?
2. Propón un entorno donde un reflejo simple sea *más* racional que un agente deliberativo
   basado en utilidad, y justifícalo con el costo de deliberación.
3. Escribe la especificación PEAS de un agente LLM con herramientas (buscador + calculadora)
   para responder preguntas factuales. ¿Cuál de los cuatro componentes es el más difícil de
   especificar y por qué?